# 06 — Live Analyser
### Real-time predictive maintenance system
> Run 00, 03, and 04 first to generate all model artifacts.
>
> **Flow:** TFT Forecaster warns if breakdown coming → FAG-TFT classifies known/unknown → Anomaly Gate detects unknown → Output with solution
>
> **States:** Healthy | Known Breakdown | Unknown Problem | Warning (Known Approaching) | Warning (Unknown Problem Approaching)

In [33]:
import pandas as pd
import numpy as np
import pickle
import time
import pyodbc
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')
try:
    import winsound
    SOUND_AVAILABLE = True
except ImportError:
    SOUND_AVAILABLE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')


✅ Device: cpu


In [34]:
# AZURE SQL CONFIGURATION
AZURE_SERVER   = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM'
AZURE_TABLE    = 'MBP_ControllerData'

# THRESHOLDS AND CONFIDENCE
MANUAL_ANOMALY_THRESHOLD = None
MANUAL_WARNING_THRESHOLD = None
CLASSIFIER_CONFIDENCE    = 0.75
FORECASTER_CONFIDENCE    = 0.60
TYPE_CONFIDENCE          = 0.50      # Model B confidence to distinguish known vs unknown forecast

print('✅ Configuration loaded.')


✅ Configuration loaded.


In [35]:
SOLUTIONS = {
    'Healthy'                                          : 'Machine operating normally. No maintenance required.',
    'High thread tension'                              : 'Reduce thread tension. Check tension discs. Verify bobbin tension. Re-thread the machine.',
    'High foot pressure'                               : 'Adjust the pressure regulator. Check presser foot spring. Calibrate foot pressure setting.',
    'Sewing without thread'                            : 'Stop machine immediately. Re-thread upper and lower threads. Check thread path for breakage.',
    'High thread tension, sewing without thread'       : 'Stop machine. Re-thread completely. Reduce thread tension after re-threading.',
    'High thread tension, sewing without bobbin thread': 'Stop machine. Replace or re-wind bobbin. Reduce upper thread tension. Check bobbin case.',
}
print(f'✅ Solutions loaded for {len(SOLUTIONS)-1} breakdown types.')


✅ Solutions loaded for 5 breakdown types.


#### Model Class Definitions


In [36]:
# GRN (used by both FAG-TFT and TFT Forecaster)
class GRN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.1):
        super().__init__()
        self.fc1=nn.Linear(input_size, hidden_size); self.fc2=nn.Linear(hidden_size, output_size)
        self.gate_fc=nn.Linear(hidden_size, output_size); self.elu=nn.ELU()
        self.sigmoid=nn.Sigmoid(); self.dropout=nn.Dropout(dropout)
        self.layer_norm=nn.LayerNorm(output_size)
        self.residual=nn.Linear(input_size, output_size) if input_size!=output_size else nn.Identity()
    def forward(self, x):
        h=self.elu(self.fc1(x)); h=self.dropout(h)
        gate=self.sigmoid(self.gate_fc(h)); out=gate*self.fc2(h)
        return self.layer_norm(out+self.residual(x))

# HierarchicalVSN (FAG-TFT only)
class HierarchicalVSN(nn.Module):
    def __init__(self, g1_size, g2_size, g3_size, el_size, hidden_size, dropout=0.1):
        super().__init__()
        self.grn_g1=GRN(g1_size,hidden_size,hidden_size,dropout); self.grn_g2=GRN(g2_size,hidden_size,hidden_size,dropout)
        self.grn_g3=GRN(g3_size,hidden_size,hidden_size,dropout); self.grn_el=GRN(el_size,hidden_size,hidden_size,dropout)
        self.sel_g1=nn.Linear(hidden_size,hidden_size); self.sel_g2=nn.Linear(hidden_size,hidden_size)
        self.sel_g3=nn.Linear(hidden_size,hidden_size); self.sel_el=nn.Linear(hidden_size,hidden_size)
        self.across_grn=GRN(hidden_size*4,hidden_size,4,dropout)
        self.group_merge=nn.Linear(hidden_size*4,hidden_size)
    def forward(self, x_g1, x_g2, x_g3, x_el):
        h1=self.grn_g1(x_g1); h2=self.grn_g2(x_g2); h3=self.grn_g3(x_g3); he=self.grn_el(x_el)
        w1=torch.sigmoid(self.sel_g1(h1)); w2=torch.sigmoid(self.sel_g2(h2))
        w3=torch.sigmoid(self.sel_g3(h3)); we=torch.sigmoid(self.sel_el(he))
        h1,h2,h3,he=w1*h1,w2*h2,w3*h3,we*he
        concat=torch.cat([h1,h2,h3,he],dim=-1)
        g_weights=F.softmax(self.across_grn(concat),dim=-1)
        self.group_weights=g_weights.detach().cpu()
        merged=(g_weights[:,0:1]*h1+g_weights[:,1:2]*h2+g_weights[:,2:3]*h3+g_weights[:,3:4]*he)
        return self.group_merge(concat)+merged

# FAG-TFT Classifier
class FAG_TFT(nn.Module):
    def __init__(self, g1_size, g2_size, g3_size, el_size, time_steps, num_classes, hidden_size=64, num_heads=4, dropout=0.1):
        super().__init__()
        self.time_steps=time_steps; self.g1_size=g1_size; self.g2_size=g2_size
        self.g3_size=g3_size; self.el_size=el_size
        self.vsn=HierarchicalVSN(g1_size,g2_size,g3_size,el_size,hidden_size,dropout)
        self.lstm_encoder=nn.LSTM(hidden_size,hidden_size,2,batch_first=True,dropout=dropout)
        self.attention=nn.MultiheadAttention(hidden_size,num_heads,dropout=dropout,batch_first=True)
        self.attn_norm=nn.LayerNorm(hidden_size)
        self.post_attn_grn=GRN(hidden_size,hidden_size,hidden_size,dropout)
        self.classifier=nn.Sequential(nn.Linear(hidden_size,32),nn.ReLU(),nn.Dropout(dropout),nn.Linear(32,num_classes))
    def forward(self, x):
        vsn_out=[]
        for t in range(self.time_steps):
            xt=x[:,t,:]
            g1=xt[:,:self.g1_size]; g2=xt[:,self.g1_size:self.g1_size+self.g2_size]
            g3=xt[:,self.g1_size+self.g2_size:self.g1_size+self.g2_size+self.g3_size]; el=xt[:,self.g1_size+self.g2_size+self.g3_size:]
            vsn_out.append(self.vsn(g1,g2,g3,el))
        vsn_seq=torch.stack(vsn_out,dim=1)
        lstm_out,_=self.lstm_encoder(vsn_seq)
        attn_out,_=self.attention(lstm_out,lstm_out,lstm_out)
        attn_out=self.attn_norm(attn_out+lstm_out)
        out=self.post_attn_grn(attn_out[:,-1,:])
        return self.classifier(out)

# Anomaly Gate
class AnomalyGate(nn.Module):
    def __init__(self, time_steps, num_features, num_vib=60, num_elec=7):
        super().__init__()
        self.time_steps=time_steps; self.num_vib=num_vib; self.num_elec=num_elec
        self.enc_vib_conv=nn.Sequential(nn.Conv1d(1,32,kernel_size=3,padding=1),nn.BatchNorm1d(32),nn.ReLU(),nn.MaxPool1d(2))
        enc_vib_size=32*(num_vib//2)
        self.enc_lstm=nn.LSTM(enc_vib_size+num_elec,32,batch_first=True)
        self.bottleneck=nn.Linear(32,16)
        self.dec_lstm=nn.LSTM(16,32,batch_first=True)
        self.dec_out=nn.Linear(32,num_features)
    def encode(self, x):
        batch=x.size(0); vib_in=x[:,:,:self.num_vib]; elec_in=x[:,:,self.num_vib:]
        vib_feats=[]
        for t in range(self.time_steps):
            vt=vib_in[:,t,:].unsqueeze(1); vt=self.enc_vib_conv(vt).view(batch,-1)
            vib_feats.append(vt)
        vib_seq=torch.stack(vib_feats,dim=1)
        merged=torch.cat([vib_seq,elec_in],dim=2)
        _,(h,_)=self.enc_lstm(merged)
        return self.bottleneck(h[-1])
    def forward(self, x):
        encoded=self.encode(x); repeated=encoded.unsqueeze(1).repeat(1,self.time_steps,1)
        dec_out,_=self.dec_lstm(repeated)
        return self.dec_out(dec_out)

# TFT Forecaster (standard - for live prediction)
class TFT_Forecaster(nn.Module):
    def __init__(self, num_features, time_steps, num_classes, hidden_size=64, num_heads=4, dropout=0.3):
        super().__init__()
        self.time_steps=time_steps
        self.input_proj=GRN(num_features, hidden_size, hidden_size, dropout)
        self.lstm_encoder=nn.LSTM(hidden_size,hidden_size,2,batch_first=True,dropout=dropout)
        self.attention=nn.MultiheadAttention(hidden_size,num_heads,dropout=dropout,batch_first=True)
        self.attn_norm=nn.LayerNorm(hidden_size)
        self.post_attn_grn=GRN(hidden_size,hidden_size,hidden_size,dropout)
        self.classifier=nn.Sequential(nn.Linear(hidden_size,32),nn.ReLU(),nn.Dropout(dropout),nn.Linear(32,num_classes))
    def forward(self, x):
        proj_out=[]
        for t in range(self.time_steps):
            proj_out.append(self.input_proj(x[:,t,:]))
        proj_seq=torch.stack(proj_out,dim=1)
        lstm_out,_=self.lstm_encoder(proj_seq)
        attn_out,_=self.attention(lstm_out,lstm_out,lstm_out)
        attn_out=self.attn_norm(attn_out+lstm_out)
        out=self.post_attn_grn(attn_out[:,-1,:])
        return self.classifier(out)

print('✅ All model classes defined.')


✅ All model classes defined.


#### Load Trained Models


In [37]:
with open('scaler.pkl','rb') as f:              scaler = pickle.load(f)
with open('encoder.pkl','rb') as f:             encoder = pickle.load(f)
with open('encoder_reason.pkl','rb') as f:      encoder_reason = pickle.load(f)
with open('tft_config.pkl','rb') as f:          cfg_class = pickle.load(f)
with open('tft_forecast_config.pkl','rb') as f: cfg_fc = pickle.load(f)
with open('ae_thresholds.pkl','rb') as f:       thresholds = pickle.load(f)

ANOMALY_THRESHOLD = MANUAL_ANOMALY_THRESHOLD or thresholds['anomaly']     # p95 - critical
WARNING_THRESHOLD = MANUAL_WARNING_THRESHOLD or thresholds.get('warning', ANOMALY_THRESHOLD * 0.85)  # p85 - early warning
TIME_STEPS        = cfg_class['time_steps']
ELEC_FEATS        = ['machineVoltageMean','machineVoltageMin','machineVoltageMax',
                      'machineCurrentMean','machineCurrentMin','machineCurrentMax']

# Load Anomaly Gate
gate = AnomalyGate(TIME_STEPS, 67).to(DEVICE)
gate.load_state_dict(torch.load('best_anomaly_gate.pt', map_location=DEVICE))
gate.eval()

# Load FAG-TFT Classifier
tft = FAG_TFT(
    g1_size=cfg_class['g1_size'], g2_size=cfg_class['g2_size'],
    g3_size=cfg_class['g3_size'], el_size=cfg_class['el_size'],
    time_steps=TIME_STEPS, num_classes=cfg_class['num_classes'],
    hidden_size=cfg_class['hidden_size'], num_heads=cfg_class['num_heads']
).to(DEVICE)
tft.load_state_dict(torch.load('best_fag_tft.pt', map_location=DEVICE))
tft.eval()

# Load TFT Forecaster (Binary)
forecaster_bin = TFT_Forecaster(
    num_features=cfg_fc['num_features'], time_steps=cfg_fc['time_steps'],
    num_classes=cfg_fc['num_classes_binary'],
    hidden_size=cfg_fc['hidden_size'], num_heads=cfg_fc['num_heads']
).to(DEVICE)
forecaster_bin.load_state_dict(torch.load('best_tft_binary.pt', map_location=DEVICE))
forecaster_bin.eval()

# Load TFT Forecaster (Type)
forecaster_type = TFT_Forecaster(
    num_features=cfg_fc['num_features'], time_steps=cfg_fc['time_steps'],
    num_classes=cfg_fc['num_classes_type'],
    hidden_size=cfg_fc['hidden_size'], num_heads=cfg_fc['num_heads']
).to(DEVICE)
forecaster_type.load_state_dict(torch.load('best_tft_type.pt', map_location=DEVICE))
forecaster_type.eval()

print('✅ All models loaded.')
print(f'   Anomaly threshold (critical) : {ANOMALY_THRESHOLD:.6f}')
print(f'   Warning threshold (early)    : {WARNING_THRESHOLD:.6f}')
print(f'   Classifier classes           : {list(encoder.classes_)}')
print(f'   Forecast horizon             : {cfg_fc["H"]} events ahead')


✅ All models loaded.
   Anomaly threshold (critical) : 1.019794
   Warning threshold (early)    : 0.790980
   Classifier classes           : ['Healthy', 'High foot pressure', 'High thread tension', 'Sewing without thread']
   Forecast horizon             : 5 events ahead


#### Feature Extraction


In [38]:
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)
    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df  = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    # Compute time_gap from dateTime
    df_t = df.copy()
    df_t['dateTime'] = pd.to_datetime(df_t['dateTime'])
    time_gap = df_t['dateTime'].diff().dt.total_seconds().fillna(0).clip(upper=300)
    time_gap_df = pd.DataFrame({'time_gap': time_gap.values})
    return pd.concat([vib_df.reset_index(drop=True), elec_df, time_gap_df], axis=1)

def alert_sound(level):
    if not SOUND_AVAILABLE: return
    if level == 'CRITICAL':
        for _ in range(3): winsound.Beep(1000, 500)
    elif level == 'WARNING':
        for _ in range(2): winsound.Beep(800, 400)

print('✅ Feature extractor ready (67 features).')


✅ Feature extractor ready (67 features).


#### Live Monitoring Loop


In [44]:
SELECTED_MACHINE = input('Enter Machine Serial Number (e.g. 8D0FH22112): ').strip()

driver   = '{ODBC Driver 17 for SQL Server}'
conn_str = (f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;'
            f'DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}')
query    = (f"SELECT TOP {TIME_STEPS} * FROM {AZURE_TABLE} "
            f"WHERE machineSerialNumber = '{SELECTED_MACHINE}' ORDER BY dateTime DESC")

last_processed_time = None

try:
    with pyodbc.connect(conn_str) as conn:
        print(f'\n✅ Connected. Monitoring: {SELECTED_MACHINE}')
        print('='*65)

        while True:
            try:
                df_live = pd.read_sql(query, conn)
                df_live = df_live[df_live['machineVibration'].astype(str).str.startswith('10')]

                # Cold start padding
                while len(df_live) < TIME_STEPS:
                    df_live = pd.concat([df_live.iloc[[0]], df_live], ignore_index=True)
                df_live = df_live.iloc[:TIME_STEPS]

                ts = df_live['dateTime'].iloc[0]
                if ts == last_processed_time:
                    time.sleep(1)
                    continue
                last_processed_time = ts

                df_window = df_live.iloc[::-1].reset_index(drop=True)

                # Feature extraction
                features = extract_features(df_window)
                X_scaled = scaler.transform(features.values)
                X_input  = torch.FloatTensor(X_scaled).unsqueeze(0).to(DEVICE)

                # ── STAGE 1: TFT FORECASTER ──────────────────────────
                with torch.no_grad():
                    fc_logits = forecaster_bin(X_input)
                    fc_probs  = torch.softmax(fc_logits, dim=1).cpu().numpy()[0]
                    fc_pred   = np.argmax(fc_probs)
                    fc_conf   = fc_probs[fc_pred]

                    forecast_warning = (fc_pred == 1 and fc_conf >= FORECASTER_CONFIDENCE)

                    # If breakdown forecasted, predict type
                    fc_type_name = 'N/A'
                    fc_type_conf = 0.0
                    if forecast_warning:
                        type_logits = forecaster_type(X_input)
                        type_probs  = torch.softmax(type_logits, dim=1).cpu().numpy()[0]
                        type_idx    = np.argmax(type_probs)
                        fc_type_conf = type_probs[type_idx]
                        fc_type_name = encoder_reason.classes_[type_idx]

                # ── STAGE 2: ANOMALY GATE ────────────────────────────
                with torch.no_grad():
                    recon    = gate(X_input)
                    ae_error = float(torch.mean(torch.abs(X_input - recon)).cpu())

                # ── STAGE 3: FAG-TFT CLASSIFIER ──────────────────────
                with torch.no_grad():
                    logits     = tft(X_input)
                    probs      = torch.softmax(logits, dim=1).cpu().numpy()[0]
                    cls_idx    = np.argmax(probs)
                    confidence = probs[cls_idx]
                    prediction = encoder.classes_[cls_idx]
                    gw = tft.vsn.group_weights.numpy()[0]
                    dominant_group = ['Low Freq','Mid Freq','High Freq','Electrical'][np.argmax(gw)]

                # ── STATE DETERMINATION ──────────────────────────────
                # Priority 1: Known breakdown happening NOW
                if prediction != 'Healthy' and confidence >= CLASSIFIER_CONFIDENCE:
                    state = 'KNOWN BREAKDOWN DETECTED'; icon = '🚨'
                    alert_sound('CRITICAL')

                # Priority 2: Unknown problem happening NOW
                elif ae_error >= ANOMALY_THRESHOLD:
                    state = 'UNKNOWN PROBLEM DETECTED'; icon = '❌'
                    alert_sound('CRITICAL')

                # Priority 3: Known breakdown approaching (forecaster warns + type confident)
                elif forecast_warning and fc_type_conf >= TYPE_CONFIDENCE:
                    state = 'KNOWN BREAKDOWN APPROACHING'; icon = '⚠️'
                    alert_sound('WARNING')

                # Priority 4: Unknown problem approaching
                #   Forecaster says something is coming BUT type model is not confident
                #   AND anomaly gate error is elevated (above warning threshold)
                elif forecast_warning and fc_type_conf < TYPE_CONFIDENCE and ae_error >= WARNING_THRESHOLD:
                    state = 'UNKNOWN PROBLEM APPROACHING'; icon = '⚠️'
                    alert_sound('WARNING')

                # Priority 5: General forecast warning (forecaster warns but no anomaly support)
                elif forecast_warning:
                    state = 'BREAKDOWN FORECASTED'; icon = '⚠️'
                    alert_sound('WARNING')

                # Default: Healthy
                else:
                    state = 'HEALTHY'; icon = '✅'

                # ── OUTPUT ───────────────────────────────────────────
                print(f'\n{"-"*65}')
                current_sl_time = pd.Timestamp.now()
                record_utc_time = pd.to_datetime(ts)
                record_sl_time  = record_utc_time + pd.Timedelta(hours=5, minutes=30)

                print(f'CURRENT TIME (SL)    : {current_sl_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'RECORD TIME (UTC)    : {record_utc_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'RECORD TIME (SL)     : {record_sl_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'STATE                : {icon} {state}')
                print(f'-'*65)
                print(f'TFT FORECAST         : {"⚠️ Breakdown in next 5 presses" if forecast_warning else "✅ Safe"} ({fc_conf*100:.1f}%)')
                if forecast_warning:
                    print(f'FORECASTED TYPE      : {fc_type_name} ({fc_type_conf*100:.1f}% confidence)')
                print(f'FAG-TFT CLASSIFIER   : {prediction} ({confidence*100:.1f}% confidence)')
                print(f'ANOMALY GATE ERROR   : {ae_error:.6f}  [warning={WARNING_THRESHOLD:.4f}  critical={ANOMALY_THRESHOLD:.4f}]')
                print(f'DOMINANT FREQ GROUP  : {dominant_group}  (VSN attention)')
                print(f'GROUP WEIGHTS        : G1={gw[0]:.2f} G2={gw[1]:.2f} G3={gw[2]:.2f} Elec={gw[3]:.2f}')
                print(f'-'*65)

                if state == 'KNOWN BREAKDOWN DETECTED':
                    print(f'BREAKDOWN  : {prediction}')
                    print(f'SOLUTION   : {SOLUTIONS.get(prediction, "Contact maintenance supervisor.")}')
                elif state == 'UNKNOWN PROBLEM DETECTED':
                    print('BREAKDOWN  : Unknown - not in trained breakdown types')
                    print('ACTION     : Stop machine. Contact maintenance supervisor immediately.')
                elif state == 'KNOWN BREAKDOWN APPROACHING':
                    print(f'WARNING    : Known breakdown predicted within next 5 pedal presses')
                    print(f'TYPE       : {fc_type_name}')
                    print(f'ACTION     : Schedule immediate inspection. {SOLUTIONS.get(fc_type_name, "Contact supervisor.")}')
                elif state == 'UNKNOWN PROBLEM APPROACHING':
                    print(f'WARNING    : Machine heading towards an unrecognised problem')
                    print(f'DETAIL     : Forecaster detects approaching breakdown but type is unrecognised')
                    print(f'             Anomaly Gate confirms unusual pattern (error: {ae_error:.6f})')
                    print(f'ACTION     : Schedule inspection. Contact maintenance supervisor.')
                elif state == 'BREAKDOWN FORECASTED':
                    print(f'WARNING    : Breakdown predicted within next 5 pedal presses')
                    print(f'TYPE       : {fc_type_name}')
                    print(f'ACTION     : Schedule immediate inspection. {SOLUTIONS.get(fc_type_name, "Contact supervisor.")}')
                else:
                    print('No maintenance required.')

            except Exception as e:
                print(f'⚠️  Loop error: {e}')
            time.sleep(1)

except KeyboardInterrupt:
    print('\n🛑 Monitoring stopped.')
except Exception as e:
    print(f'\n❌ Connection error: {e}')



-----------------------------------------------------------------
CURRENT TIME (SL)    : 2026-03-30 16:32:47
RECORD TIME (UTC)    : 2026-03-30 11:02:45
RECORD TIME (SL)     : 2026-03-30 16:32:45
STATE                : ❌ UNKNOWN PROBLEM DETECTED
-----------------------------------------------------------------
TFT FORECAST         : ✅ Safe (56.4%)
FAG-TFT CLASSIFIER   : Healthy (27.6% confidence)
ANOMALY GATE ERROR   : 1.054807  [warning=0.7910  critical=1.0198]
DOMINANT FREQ GROUP  : Low Freq  (VSN attention)
GROUP WEIGHTS        : G1=0.74 G2=0.12 G3=0.06 Elec=0.08
-----------------------------------------------------------------
BREAKDOWN  : Unknown - not in trained breakdown types
ACTION     : Stop machine. Contact maintenance supervisor immediately.

-----------------------------------------------------------------
CURRENT TIME (SL)    : 2026-03-30 16:32:53
RECORD TIME (UTC)    : 2026-03-30 11:02:51
RECORD TIME (SL)     : 2026-03-30 16:32:51
STATE                : ❌ UNKNOWN PROBLEM